# Capstone: Build a mini-GPT from scratch

_Capstones_

**Seven landmark papers, built in order, become one small character-level GPT that generates text.**

---

This notebook is a practice scaffold for the **Capstone: Build a mini-GPT from scratch** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# ============================================================================
# THE FINAL BUILD: a char-level nanoGPT assembled from the 7 capstone papers.
#   paper-word2vec    -> token embedding table        (nn.Embedding below)
#   paper-attention   -> scaled dot-product attention (the Q@K^T/sqrt(d_k) softmax)
#   paper-transformer -> multi-head attention + positions + the block
#   paper-layernorm   -> nn.LayerNorm inside each block
#   paper-resnet      -> the "x + sublayer(x)" residual additions
#   paper-adamw       -> torch.optim.AdamW (decoupled weight decay)
#   paper-gpt         -> causal mask + next-token cross-entropy + sampling
# ============================================================================


# === 1. Causal multi-head self-attention (paper-attention + paper-transformer + paper-gpt's mask). ===
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, h, max_len):
        super().__init__()
        assert d_model % h == 0
        self.h, self.d_k = h, d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # paper-gpt's causal mask: True ABOVE the diagonal = the future, which we forbid.
        self.register_buffer("mask", torch.triu(torch.ones(max_len, max_len), diagonal=1).bool())

    def _split(self, x):                                    # (B,S,d_model) -> (B,h,S,d_k): the "heads"
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, S, _ = x.shape
        Q, K, V = self._split(self.W_q(x)), self._split(self.W_k(x)), self._split(self.W_v(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)         # scaled dot product (paper-attention)
        scores = scores.masked_fill(self.mask[:S, :S], float("-inf"))  # forbid the future (paper-gpt)
        out = F.softmax(scores, dim=-1) @ V                            # attention weights @ values
        out = out.transpose(1, 2).contiguous().view(B, S, self.h * self.d_k)  # concat the heads
        return self.W_o(out)


# === 2. Pre-norm block: LayerNorm -> sublayer -> residual add (paper-layernorm + paper-resnet). ===
class Block(nn.Module):
    def __init__(self, d_model, h, max_len, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)                    # paper-layernorm (pre-norm: LN at the input)
        self.attn = CausalSelfAttention(d_model, h, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # residual around masked attention (paper-resnet)
        x = x + self.ff(self.ln2(x))       # residual around feed-forward    (paper-resnet)
        return x


# === 3. The nanoGPT: token + learned positional embeddings, N blocks, final LN, vocab head. ===
class NanoGPT(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, n_layers=3, max_len=64, d_ff=128):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)             # token embedding table (paper-word2vec)
        self.pos = nn.Embedding(max_len, d_model)           # LEARNED positional embedding (paper-transformer)
        self.blocks = nn.ModuleList([Block(d_model, h, max_len, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)                   # GPT-2's extra final LayerNorm
        self.head = nn.Linear(d_model, vocab)               # the language-model head -> per-char logits

    def forward(self, idx):
        B, S = idx.shape
        pos = torch.arange(S, device=idx.device)
        x = self.tok(idx) + self.pos(pos)                   # add token + positional embeddings
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln_f(x))                      # (B,S,vocab) next-char logits

    @torch.no_grad()
    def generate(self, idx, n_new, temp=0.8, max_len=64):
        for _ in range(n_new):
            logits = self(idx[:, -max_len:])[:, -1, :] / temp   # logits for the next char at the last slot
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)               # SAMPLE the next char (paper-gpt)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


# === 4. Tiny CHAR-level corpus. Target = input shifted left by one (predict the NEXT char). ===
text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
VOCAB = len(chars)                                          # ~20 here -> random-init loss ~ ln(20) ~ 3.0
data = torch.tensor([stoi[c] for c in text])
SEQ, B = 32, 64

def get_batch():
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ] for i in ix])
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])  # NEXT char at every position (shift by 1)
    return x, y

def sample(net):
    start = torch.tensor([[stoi["t"]]])
    out = net.generate(start, 60)[0].tolist()
    return "".join(itos[i] for i in out)


# === 5. Train with AdamW on next-char cross-entropy; print loss + a sample as it improves. ===
torch.manual_seed(0)
net = NanoGPT(VOCAB, max_len=SEQ)
opt = torch.optim.AdamW(net.parameters(), lr=3e-3)          # paper-adamw: decoupled weight decay
for step in range(2001):
    x, y = get_batch()
    logits = net(x)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))  # causal-LM objective (paper-gpt)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}   sample: {sample(net)!r}")
# Loss falls from ~3.0 (random over ~20 chars) toward ~0.2, and samples go gibberish -> readable.
# Our small run, not a paper's reported number; exact values vary by hardware and seed.

## Visualize the data & results

_As the assembled char-level mini-GPT trains with AdamW on the next-character cross-entropy objective, does the loss fall and does the generated text become readable?_

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

# The same assembled mini-GPT as the final build, run once to record loss + sample-quality curves.
class CSA(nn.Module):                                   # causal self-attention
    def __init__(self, d, h, mx):
        super().__init__(); self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))
        self.register_buffer("m", torch.triu(torch.ones(mx, mx), 1).bool())
    def split(self, x):
        B, S, _ = x.shape; return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, x):
        B, S, _ = x.shape
        Q, K, V = self.split(self.Wq(x)), self.split(self.Wk(x)), self.split(self.Wv(x))
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        sc = sc.masked_fill(self.m[:S, :S], float("-inf"))   # forbid the future
        a = F.softmax(sc, dim=-1) @ V
        return self.Wo(a.transpose(1, 2).contiguous().view(B, S, self.h * self.dk))

class Block(nn.Module):
    def __init__(self, d, h, mx, ff):
        super().__init__()
        self.n1, self.a, self.n2 = nn.LayerNorm(d), CSA(d, h, mx), nn.LayerNorm(d)
        self.f = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
    def forward(self, x):
        x = x + self.a(self.n1(x)); return x + self.f(self.n2(x))

class GPT(nn.Module):
    def __init__(self, V, d=64, h=4, L=3, mx=32, ff=128):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(mx, d)
        self.b = nn.ModuleList([Block(d, h, mx, ff) for _ in range(L)])
        self.lnf, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, idx):
        B, S = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(S))
        for blk in self.b: x = blk(x)
        return self.head(self.lnf(x))

text = ("to be, or not to be, that is the question. "
        "whether tis nobler in the mind to suffer. ") * 50
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
V = len(chars); data = torch.tensor([stoi[c] for c in text]); SEQ, B = 32, 64
net = GPT(V, mx=SEQ); opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
for step in range(2001):
    ix = torch.randint(0, len(data) - SEQ - 1, (B,))
    x = torch.stack([data[i:i + SEQ] for i in ix])
    y = torch.stack([data[i + 1:i + SEQ + 1] for i in ix])
    loss = F.cross_entropy(net(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0: print(step, round(loss.item(), 3))
# Record (step, loss) and (step, readable-char fraction of a generated sample) -> the two panels above.
# Our small run, not a paper's reported number.